In [ ]:
"""
PII Detection and Masking Training Pipeline
-------------------------------------------

This script trains a transformer-based Named Entity Recognition (NER) model
to detect and mask Personally Identifiable Information (PII) in text.
The model focuses on two entity types: PERSON names and EMAIL addresses.

A DeBERTa-v3-small transformer is fine-tuned for token classification
using BIO tagging. Class-weighted loss is applied to address label
imbalance, and early stopping is used to prevent overfitting.

Pipeline Overview:
1. Load processed training and test datasets
2. Convert data into HuggingFace Dataset format
3. Tokenize text and align NER labels with subword tokens
4. Train the DeBERTa NER model with weighted loss
5. Evaluate performance using precision, recall, F1, FPR, and FNR
6. Perform PII masking on the test dataset
7. Save the trained model and tokenizer

The final output includes evaluation metrics, masked text predictions,
and a downloadable trained model.
"""

# Install required libraries
!pip install evaluate seqeval

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import evaluate

from tqdm.auto import tqdm
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
import shutil


# -----------------------------------------------------------
# 1. CONFIGURATION
# -----------------------------------------------------------

TRAIN_PATH = "/kaggle/input/datasets/abdullahshheikh/pii-masking-augmented/train_processed.json"
TEST_PATH = "/kaggle/input/datasets/abdullahshheikh/pii-masking-synthetic-data/email_test_robust.json"

MODEL_CHECKPOINT = "microsoft/deberta-v3-small"

label_list = ["O", "B-PER", "I-PER", "B-EMAIL", "I-EMAIL"]
label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label = {i: l for i, l in enumerate(label_list)}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------------------------------------
# 2. DATA LOADING
# -----------------------------------------------------------

with open(TRAIN_PATH, "r") as f:
    train_data = json.load(f)

with open(TEST_PATH, "r") as f:
    test_data = json.load(f)

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# Split training data into train + validation
split_ds = train_dataset.train_test_split(test_size=0.1, seed=42)

raw_datasets = DatasetDict({
    "train": split_ds["train"],
    "validation": split_ds["test"],
    "test": test_dataset
})


# -----------------------------------------------------------
# 3. TOKENIZATION & LABEL ALIGNMENT
# -----------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def align_labels(examples):
    """
    Align word-level NER labels with tokenized subword tokens.
    """
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word:
                label_ids.append(label_to_id[label[word_idx]])

            else:
                full_label = label[word_idx]

                if full_label.startswith("B-"):
                    new_label = "I-" + full_label[2:]
                    label_ids.append(label_to_id[new_label])
                else:
                    label_ids.append(label_to_id[full_label])

            previous_word = word_idx

        labels.append(label_ids)

    tokenized["labels"] = labels
    return tokenized


tokenized_ds = raw_datasets.map(align_labels, batched=True)


# -----------------------------------------------------------
# 4. EVALUATION METRICS
# -----------------------------------------------------------

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    """
    Computes token classification metrics including
    precision, recall, F1, FPR and FNR for PER and EMAIL.
    """

    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = [
        [id_to_label[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labs = [
        [id_to_label[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    flat_preds = [t for seq in true_preds for t in seq]
    flat_labs = [t for seq in true_labs for t in seq]

    metrics = {}

    for tag in ["PER", "EMAIL"]:

        y_true = [1 if tag in t else 0 for t in flat_labs]
        y_pred = [1 if tag in t else 0 for t in flat_preds]

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

        metrics[f"{tag}_precision"] = precision_score(y_true, y_pred, zero_division=0)
        metrics[f"{tag}_recall"] = recall_score(y_true, y_pred, zero_division=0)
        metrics[f"{tag}_f1"] = f1_score(y_true, y_pred, zero_division=0)

        metrics[f"{tag}_FPR"] = fp / (fp + tn) if (fp + tn) > 0 else 0
        metrics[f"{tag}_FNR"] = fn / (fn + tp) if (fn + tp) > 0 else 0

    seq_results = seqeval.compute(
        predictions=true_preds,
        references=true_labs
    )

    metrics["overall_precision"] = seq_results["overall_precision"]
    metrics["overall_recall"] = seq_results["overall_recall"]
    metrics["overall_f1"] = seq_results["overall_f1"]
    metrics["overall_accuracy"] = seq_results["overall_accuracy"]

    return metrics


# -----------------------------------------------------------
# 5. MODEL & TRAINING
# -----------------------------------------------------------

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

model.config.layer_norm_eps = 1e-6


class_weights = torch.tensor([1.0, 3.0, 3.0, 6.0, 4.0]).to(device)


class WeightedTrainer(Trainer):
    """
    Custom Trainer that applies class-weighted loss
    to handle label imbalance.
    """

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs["labels"]

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights,
            ignore_index=-100
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(

    output_dir="./pii_deberta_results",

    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,

    num_train_epochs=10,
    learning_rate=1e-5,

    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    max_grad_norm=1.0,

    evaluation_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    logging_steps=50
)


trainer = WeightedTrainer(

    model=model,
    args=training_args,

    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],

    data_collator=DataCollatorForTokenClassification(tokenizer),

    compute_metrics=compute_metrics,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)


model.gradient_checkpointing_enable()

trainer.train()


# -----------------------------------------------------------
# 6. FINAL EVALUATION
# -----------------------------------------------------------

test_metrics = trainer.evaluate(tokenized_ds["test"])
print(test_metrics)


# -----------------------------------------------------------
# 7. PII MASKING EVALUATION
# -----------------------------------------------------------

def run_pii_masking_assessment(trainer, dataset, tokenizer):

    model = trainer.model
    model.eval()
    model.to(device)

    stats = {
        "PER": {"tp":0,"fp":0,"tn":0,"fn":0},
        "EMAIL": {"tp":0,"fp":0,"tn":0,"fn":0}
    }

    masked_results = []

    for sample in tqdm(dataset):

        inputs = {
            "input_ids": torch.tensor(sample["input_ids"]).unsqueeze(0).to(device),
            "attention_mask": torch.tensor(sample["attention_mask"]).unsqueeze(0).to(device)
        }

        with torch.no_grad():
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=-1).squeeze().cpu().tolist()

        tokens = sample["tokens"]
        true_labels = sample["ner_tags"]

        encoding = tokenizer(tokens, is_split_into_words=True)
        word_ids = encoding.word_ids()

        predicted_labels = []
        prev = None

        for idx, word_idx in enumerate(word_ids):
            if word_idx is None:
                continue

            if word_idx != prev:
                predicted_labels.append(id_to_label[preds[idx]])
                prev = word_idx

        predicted_labels = predicted_labels[:len(tokens)]

        masked_tokens = []

        for token, pred in zip(tokens, predicted_labels):

            if "PER" in pred:
                masked_tokens.append("[NAME]")
            elif "EMAIL" in pred:
                masked_tokens.append("[EMAIL]")
            else:
                masked_tokens.append(token)

        masked_results.append({
            "original": " ".join(tokens),
            "masked": " ".join(masked_tokens),
            "predictions": predicted_labels
        })

        for true_tag, pred_tag in zip(true_labels, predicted_labels):

            for cat in ["PER","EMAIL"]:

                is_true = cat in true_tag
                is_pred = cat in pred_tag

                if is_true and is_pred:
                    stats[cat]["tp"] += 1
                elif not is_true and is_pred:
                    stats[cat]["fp"] += 1
                elif is_true and not is_pred:
                    stats[cat]["fn"] += 1
                else:
                    stats[cat]["tn"] += 1

    report = []

    for cat in ["PER","EMAIL"]:

        s = stats[cat]

        precision = s["tp"]/(s["tp"]+s["fp"]) if (s["tp"]+s["fp"]) else 0
        recall = s["tp"]/(s["tp"]+s["fn"]) if (s["tp"]+s["fn"]) else 0
        accuracy = (s["tp"]+s["tn"])/(s["tp"]+s["tn"]+s["fp"]+s["fn"])
        fpr = s["fp"]/(s["fp"]+s["tn"]) if (s["fp"]+s["tn"]) else 0
        fnr = s["fn"]/(s["fn"]+s["tp"]) if (s["fn"]+s["tp"]) else 0
        f1 = 2*(precision*recall)/(precision+recall) if (precision+recall) else 0

        report.append({
            "Entity": cat,
            "Accuracy": round(accuracy,4),
            "Precision": round(precision,4),
            "Recall": round(recall,4),
            "F1": round(f1,4),
            "FPR": round(fpr,4),
            "FNR": round(fnr,4)
        })

    return pd.DataFrame(report), masked_results


metrics_df, masked_output = run_pii_masking_assessment(
    trainer,
    tokenized_ds["test"],
    tokenizer
)

print(metrics_df)


# -----------------------------------------------------------
# 8. SAVE MODEL
# -----------------------------------------------------------

save_dir = "/kaggle/working/final_pii_model"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

shutil.make_archive("pii_model_download", "zip", save_dir)

print("Model saved and zipped successfully.")

Overlapping sentences: 0
--- RAW DATA CHECK ---
Total tokens: 731505
Label distribution: {np.str_('B-EMAIL'): np.int64(11416), np.str_('B-PER'): np.int64(40264), np.str_('I-PER'): np.int64(29466), np.str_('O'): np.int64(650359)}


Map:   0%|          | 0/25664 [00:00<?, ? examples/s]

Map:   0%|          | 0/2852 [00:00<?, ? examples/s]

Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

--- TRAINING TAG DISTRIBUTION ---
Tag        | Count      | Percentage
-----------------------------------
O          | 620610     |  77.7967%
B-PER      | 36286      |   4.5486%
I-PER      | 53797      |   6.7437%
B-EMAIL    | 10243      |   1.2840%
I-EMAIL    | 76797      |   9.6269%

--- VALIDATION TAG DISTRIBUTION ---
Tag        | Count      | Percentage
-----------------------------------
O          | 69358      |  77.9654%
B-PER      | 3978       |   4.4717%
I-PER      | 5689       |   6.3950%
B-EMAIL    | 1173       |   1.3186%
I-EMAIL    | 8762       |   9.8494%


...

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


--- FINAL METRICS ON UNSEEN TEST DATA ---



--- 🧪 DEBUG: PREDICTION CHECK (Step 9624) ---
Token           | True         | Pred         | PII Prob (%)
------------------------------------------------------------
▁Edmund         | B-PER        | B-PER        | 100.00%
▁Spenser        | I-PER        | I-PER        | 100.00%
{'eval_loss': 1.1431167125701904, 'eval_PER_precision': 0.9916418756197761, 'eval_PER_recall': 1.0, 'eval_PER_f1': 0.9958033999573227, 'eval_PER_FPR': 0.0008636210606437636, 'eval_PER_FNR': 0.0, 'eval_EMAIL_precision': 0.9987736152071715, 'eval_EMAIL_recall': 0.8578271555399508, 'eval_EMAIL_f1': 0.9229503109779954, 'eval_EMAIL_FPR': 0.0011850012696442175, 'eval_EMAIL_FNR': 0.14217284446004916, 'eval_overall_precision': 0.5984197393956197, 'eval_overall_recall': 0.616978705159354, 'eval_overall_f1': 0.6075575258602491, 'eval_overall_accuracy': 0.899703918106138, 'eval_runtime': 7.201, 'eval_samples_per_second': 486.046, 'eval_steps_per_second': 60.825, 'epoch': 6.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

--- SUCCESS ---
Model saved to: /kaggle/working/final_pii_model
ZIP file created: /kaggle/working/pii_model_download.zip
